# 🚌 SUNU BRT DAKAR - SYSTÈME INTÉGRÉ DE PILOTAGE STRATÉGIQUE

Structure du notebook:
1. ETL - Nettoyage et normalisation
2. EDA - Audit qualité données
3. KPIs - Métriques stratégiques
4. Dashboard Exécutif
5. Visualisations

## ETL - Extraction, Nettoyage, Normalisation

In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display, Markdown

DATA_PATH = r'C:\Users\7MAKSACOD\OneDrive\Desktop\brt\brt data.xlsx'
xls = pd.ExcelFile(DATA_PATH)
sheets = {s: pd.read_excel(xls, s) for s in xls.sheet_names}

def professional_etl(sheets):
    for s_name in ['Fact_Frequentation', 'Fact_Ponctualite', 'Fact_TempsArret']:
        df = sheets[s_name]
        d_str = df['DateCalendrier'].astype(str).str.split(' ').str[0]
        h_str = df['Heure'].astype(str).str.strip()
        df['ts'] = pd.to_datetime(d_str + ' ' + h_str, errors='coerce')
        df = df.dropna(subset=['ts']).drop_duplicates()
        df['hour'] = df['ts'].dt.hour
        df['day_type'] = df['ts'].dt.dayofweek.apply(lambda x: 'Weekend' if x >= 5 else 'ouvre')
        df['peak_flag'] = df['hour'].apply(lambda x: 'Peak' if (7<=x<=9 or 16<=x<=19) else 'OffPeak')
        sheets[s_name] = df
        print(f'Table {s_name}: {len(df)} rows')
    return sheets

clean_sheets = professional_etl(sheets)

Table Fact_Frequentation: 100000 rows
Table Fact_Ponctualite: 10000 rows
Table Fact_TempsArret: 10000 rows


## EDA - Audit Qualité

In [3]:
quality_audit = []
for name, df in clean_sheets.items():
    quality_audit.append({
        'Table': name, 
        'Doublons': df.duplicated().sum(),
        'Valeurs Manquantes': df.isnull().sum().sum()
    })
display(pd.DataFrame(quality_audit))
print('Audit qualite termine')

,Table,Doublons,Valeurs Manquantes
0,Dim_Calendrier,0,0
1,Dim_TrancheHoraire,0,0
2,Dim_Station,0,0
3,Dim_Ligne,0,0
4,Dim_Vehicule,0,0
5,Dim_Chauffeur,0,0
6,Fact_Frequentation,0,0
7,Fact_Ponctualite,0,0
8,Fact_TempsArret,0,0
9,Fact_Trajets,0,0


Audit qualite termine


## Master Dataset - Jointures

In [4]:
def build_master(sheets):
    for s in ['Fact_Frequentation', 'Fact_Ponctualite', 'Fact_TempsArret', 'Dim_Station']:
        if 'IDStation' in sheets[s].columns:
            sheets[s]['IDStation'] = sheets[s]['IDStation'].astype(str).str.strip()

    keys = ['ts', 'IDStation', 'IDLigne', 'IDBus']
    master = pd.merge(sheets['Fact_Frequentation'], sheets['Fact_Ponctualite'][keys + ['RetardMinutes', 'EstPonctuel']], on=keys, how='left')
    master = pd.merge(master, sheets['Fact_TempsArret'][keys + ['TempsArretSecondes']], on=keys, how='left')
    master = pd.merge(master, sheets['Dim_Station'], on='IDStation', how='left')
    return master

master = build_master(clean_sheets)
print(f'Master Dataset: {master.shape[0]} lignes')

Master Dataset: 100000 lignes


## KPIs - Métriques Stratégiques

In [5]:
# KPIs directs depuis les tables sources
ponct = clean_sheets['Fact_Ponctualite']
freq = clean_sheets['Fact_Frequentation']

otp_global = (ponct['EstPonctuel'] == 'Oui').mean() * 100
avg_delay = ponct['RetardMinutes'].mean()
total_pax = freq['Montees'].sum()
avg_load = freq['TauxRemplissage'].mean()
load_factor = avg_load * 100
revenue = total_pax * 350
co2_avoided = total_pax * 0.15 / 1000

print('=== KPI GLOBAUX ===')
print(f'OTP: {otp_global:.1f}%')
print(f'Retard: {avg_delay:.1f} min')
print(f'Passagers: {total_pax:,}')
print(f'Load: {load_factor:.1f}%')
print(f'Recettes: {revenue/1e6:.1f}M')
print(f'CO2: {co2_avoided:.1f}t')

=== KPI GLOBAUX ===
OTP: 72.5%
Retard: 3.8 min
Passagers: 6,453,918
Load: 5859.4%
Recettes: 2258.9M
CO2: 968.1t


## Dashboard Exécutif

In [6]:
exec_kpis = pd.DataFrame({
    'KPI': ['OTP', 'Passagers', 'Load Factor', 'Recettes', 'CO2'],
    'Valeur': [f'{otp_global:.1f}%', f'{total_pax:,}', f'{load_factor:.1f}%', f'{revenue/1e6:.1f}M', f'{co2_avoided:.1f}t'],
    'Cible': ['>90%', '85000', '75%', '30M', '-'],
})
display(Markdown('### Executive Summary'))
display(exec_kpis)

### Executive Summary

,KPI,Valeur,Cible
0,OTP,72.5%,>90%
1,Passagers,"6,453,918",85000
2,Load Factor,5859.4%,75%
3,Recettes,2258.9M,30M
4,CO2,968.1t,-


## Top Stations

In [7]:
station_stats = freq.groupby('IDStation').agg({
    'Montees': 'sum',
    'TauxRemplissage': 'mean'
}).reset_index()

station_stats = station_stats.merge(
    clean_sheets['Dim_Station'][['IDStation', 'NomStation']],
    on='IDStation', how='left'
)
station_stats = station_stats.sort_values('Montees', ascending=False)
station_stats['load_pct'] = station_stats['TauxRemplissage'] * 100

print('=== TOP 15 STATIONS ===')
for _, row in station_stats.head(15).iterrows():
    print(f"{row.get('NomStation','N/A'):40} {row['Montees']:>10} pax")

=== TOP 15 STATIONS ===
Petersen / Papa Gueye Fall                   678007 pax
Préfecture de Guédiawaye                     675636 pax
Grand Médine                                 524571 pax
Dial Diop / Thiandoum                        277532 pax
Dalal Jamm / Hôpital Dalal Jamm              275555 pax
Malika                                       273277 pax
Keur Massar                                  271249 pax
Yeumbeul                                     270585 pax
Sacré-Cœur / Liberté                         269785 pax
Place de la Nation / Baux Maraîchers         265748 pax
Khar Yallah                                  212989 pax
Golf Nord                                    212400 pax
Thiaroye                                     208954 pax
Golf                                         207212 pax
Parcelles Assainies                          206856 pax


## Performance par Ligne

In [8]:
for lid in [1, 2, 3]:
    l_ponct = ponct[ponct['IDLigne'] == lid]
    l_freq = freq[freq['IDLigne'] == lid]
    l_otp = (l_ponct['EstPonctuel'] == 'Oui').mean() * 100
    l_pax = l_freq['Montees'].sum()
    l_delay = l_ponct['RetardMinutes'].mean()
    print(f'Ligne {lid}: OTP={l_otp:.1f}%, Pax={l_pax:,}, Delay={l_delay:.1f}min')

Ligne 1: OTP=71.4%, Pax=2,165,004, Delay=3.8min
Ligne 2: OTP=73.0%, Pax=2,142,234, Delay=3.7min
Ligne 3: OTP=73.2%, Pax=2,146,680, Delay=3.8min


## Visualisations - Jauges

In [9]:
from plotly.subplots import make_subplots

fig_g = make_subplots(rows=1, cols=3, specs=[[{'type':'indicator'},{'type':'indicator'},{'type':'indicator'}]])

fig_g.add_trace(go.Indicator(mode='gauge+number', value=otp_global, title={'text':'OTP (%)'},
    gauge={'axis':{'range':[0,100]}, 'bar':{'color':'#1D9E75'},
           'steps':[{'range':[0,85],'color':'#FF4C4C'},{'range':[85,90],'color':'#FFA726'},{'range':[90,100],'color':'#E8F5E9'}]}), row=1, col=1)

fig_g.add_trace(go.Indicator(mode='gauge+number', value=load_factor, title={'text':'Load Factor (%)'},
    gauge={'axis':{'range':[0,100]}, 'bar':{'color':'#1A6FA4'}}), row=1, col=2)

fig_g.add_trace(go.Indicator(mode='gauge+number', value=avg_delay, title={'text':'Retard (min)'},
    gauge={'axis':{'range':[0,15]}, 'bar':{'color':'#F59E0B'}}), row=1, col=3)

fig_g.update_layout(template='plotly_dark', height=300)
display(fig_g)

## Heatmap Fréquentation

In [10]:
pivot_freq = freq.groupby(['IDStation', 'hour'])['Montees'].sum().reset_index()
pivot_freq = pivot_freq.merge(clean_sheets['Dim_Station'][['IDStation', 'NomStation']], on='IDStation', how='left')
pivot_pivot = pivot_freq.pivot_table(index='NomStation', columns='hour', values='Montees', fill_value=0)

fig_heat = px.imshow(pivot_pivot, color_continuous_scale='YlOrRd', title='Heatmap Station x Heure')
fig_heat.update_layout(template='plotly_dark', height=700)
display(fig_heat)

## Top Stations Visualisation

In [11]:
top_st = station_stats.head(10).copy()
fig_top = px.bar(top_st, x='Montees', y='NomStation', orientation='h', title='Top 10 Stations')
fig_top.update_layout(template='plotly_dark', height=500)
display(fig_top)

## Profil Horaire

In [12]:
hourly = freq.groupby('hour')['Montees'].sum().reset_index()
hourly['peak'] = hourly['hour'].apply(lambda x: 'Pointe' if (7<=x<=9 or 16<=x<=19) else 'HorsPointe')

fig_h = px.bar(hourly, x='hour', y='Montees', color='peak', title='Profil Horaire')
fig_h.update_layout(template='plotly_dark', height=400)
display(fig_h)

## Performance par Ligne - Graphique

In [13]:
line_kpi = pd.DataFrame()
for lid in [1, 2, 3]:
    l_ponct = ponct[ponct['IDLigne'] == lid]
    l_freq = freq[freq['IDLigne'] == lid]
    l_otp = (l_ponct['EstPonctuel'] == 'Oui').mean() * 100
    l_pax = l_freq['Montees'].sum()
    l_delay = l_ponct['RetardMinutes'].mean()
    line_kpi = pd.concat([line_kpi, pd.DataFrame({
        'Ligne': [f'Ligne {lid}'], 'OTP': [l_otp], 'Pax': [l_pax], 'Delay': [l_delay]
    })])

fig_line = px.bar(line_kpi.melt(id_vars='Ligne', var_name='KPI', value_name='Value'),
    x='Ligne', y='Value', color='KPI', title='Performance par Ligne', barmode='group')
fig_line.update_layout(template='plotly_dark', height=400)
display(fig_line)

## Finance - Analyse des Recettes et Coûts

In [14]:
# Analyse Finance
revenue_total = total_pax * 350  # 350 FCA par passager
cout_energie = revenue_total * 0.24
cout_rh = revenue_total * 0.33
cout_maint = revenue_total * 0.15
cout_autres = revenue_total * 0.09
marge = revenue_total - cout_energie - cout_rh - cout_maint - cout_autres

print('=== ANALYSE FINANCIÈRE ===')
print(f'Recettes totales: {revenue_total/1e6:.1f}M FCA')
print(f'  Energie (24%): {cout_energie/1e6:.1f}M')
print(f'  RH (33%): {cout_rh/1e6:.1f}M')
print(f'  Maintenance (15%): {cout_maint/1e6:.1f}M')
print(f'  Autres (9%): {cout_autres/1e6:.1f}M')
print(f'Marge nette: {marge/1e6:.1f}M FCA')
print(f'Taux de marge: {marge/revenue_total*100:.1f}%')

=== ANALYSE FINANCIÈRE ===
Recettes totales: 2258.9M FCA
  Energie (24%): 542.1M
  RH (33%): 745.4M
  Maintenance (15%): 338.8M
  Autres (9%): 203.3M
Marge nette: 429.2M FCA
Taux de marge: 19.0%


In [15]:
# Waterfall Finance
waterfall = pd.DataFrame([
    {'Category': 'Recettes', 'Value': revenue_total/1e6, 'Type': 'positive'},
    {'Category': 'Energie', 'Value': -cout_energie/1e6, 'Type': 'negative'},
    {'Category': 'RH', 'Value': -cout_rh/1e6, 'Type': 'negative'},
    {'Category': 'Maintenance', 'Value': -cout_maint/1e6, 'Type': 'negative'},
    {'Category': 'Autres', 'Value': -cout_autres/1e6, 'Type': 'negative'},
    {'Category': 'Marge', 'Value': marge/1e6, 'Type': 'positive'},
])

fig_water = go.Figure(go.Waterfall(
    name='Finance', orientation='v',
    measure=['relative', 'relative', 'relative', 'relative', 'relative', 'total'],
    x=waterfall['Category'],
    textposition='outside',
    text=[f'{v:.1f}M' for v in waterfall['Value']],
    y=waterfall['Value'],
    connector={'line': {'color': 'rgb(63, 63, 63)'}},
))
fig_water.update_layout(title='Waterfall Finance (Millions FCA)', template='plotly_dark')
display(fig_water)

## Flotte - État du Parc

In [16]:
# Analyse Flotte
vehicules = clean_sheets['Dim_Vehicule']
print('=== STATUT FLOTTE ===')
print(f'Total vehicules: {len(vehicules)}')
for status in ['Actif', 'Maintenance', 'Hors service']:
    count = len(vehicules[vehicules['Statut'] == status])
    pct = count/len(vehicules)*100
    print(f'  {status}: {count} ({pct:.1f}%)')

# Age moyen
vehicules['Annee'] = pd.to_numeric(vehicules['Annee'], errors='coerce')
avg_year = vehicules['Annee'].mean()
print(f'Annee moyenne: {2026-avg_year:.1f} ans')

=== STATUT FLOTTE ===
Total vehicules: 150
  Actif: 140 (93.3%)
  Maintenance: 10 (6.7%)
  Hors service: 0 (0.0%)


KeyError: 'Annee'

## RH - Chauffeurs

In [17]:
# Analyse RH
chauffeurs = clean_sheets['Dim_Chauffeur']
print('=== EFFECTIF CHAUFFEURS ===')
print(f'Total chauffeurs: {len(chauffeurs)}')
for status in ['Actif', 'Inactif']:
    count = len(chauffeurs[chauffeurs['Statut'] == status])
    pct = count/len(chauffeurs)*100
    print(f'  {status}: {count} ({pct:.1f}%)')

# Anciennete moyenne
chauffeurs['Anciennete'] = pd.to_numeric(chauffeurs['Anciennete'], errors='coerce')
avg_anc = chauffeurs['Anciennete'].mean()
print(f'Anciennete moyenne: {avg_anc:.1f} ans')

=== EFFECTIF CHAUFFEURS ===
Total chauffeurs: 200
  Actif: 200 (100.0%)
  Inactif: 0 (0.0%)
Anciennete moyenne: 1.4 ans


## Alertes et Actions Prioritaires

In [18]:
# Generation alertes
alerts = []

# Alert OTP
if otp_global < 85:
    alerts.append({'Severity': 'critical', 'Title': f'OTP bas: {otp_global:.1f}%', 'Action': 'Audit causes retards'})

# Alert load factor
if load_factor > 90:
    alerts.append({'Severity': 'warning', 'Title': 'Stations surchargees', 'Action': 'Augmenter freq'})

# Alert stations
for _, row in critical_stations.head(3).iterrows():
    alerts.append({'Severity': 'warning', 'Title': f'Surcharge: {row.get("NomStation","N/A")}', 'Action': 'Monitoring accru'})

print('=== ALERTES PRIORITAIRES ===')
for a in alerts:
    print(f'  [{a["Severity"]}] {a["Title"]} - {a["Action"]}')

NameError: name 'critical_stations' is not defined

## Résumé Exécutif

In [19]:
# Resume final
print('='*50)
print('RÉSUMÉ EXÉCUTIF - SUNU BRT DAKAR')
print('='*50)
print(f'OTP Global: {otp_global:.1f}% (Cible: >90%)')
print(f'Passagers total: {total_pax:,}')
print(f'Recettes: {revenue_total/1e6:.1f}M FCA')
print(f'Marge: {marge/1e6:.1f}M FCA ({marge/revenue_total*100:.1f}%)')
print(f'Flotte: {len(vehicules)} bus')
print(f'Chauffeurs: {len(chauffeurs)}')
print('='*50)

RÉSUMÉ EXÉCUTIF - SUNU BRT DAKAR
OTP Global: 72.5% (Cible: >90%)
Passagers total: 6,453,918
Recettes: 2258.9M FCA
Marge: 429.2M FCA (19.0%)
Flotte: 150 bus
Chauffeurs: 200
